In [ ]:
import os
import torch
from datasets import load_dataset
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [ ]:
repo_id = 'microsoft/Phi-3-mini-4k-instruct'
model = AutoModelForCausalLM.from_pretrained(repo_id, device_map='cuda:0', quantization_config=bnb_config)

In [ ]:
print(model.get_memory_footprint()/1000000)

In [ ]:
model

In [ ]:
model = prepare_model_for_kbit_training(model)

config = LoraConfig(
    r=8, # Entre mas pequeño menos parametros necesitara entrenar.
    lora_alpha=16, # multiplicador, normalmente debe ser 2*r
    bias='none',
    lora_dropout=0.05,
    task_type='CAUSAL_LM',
    # Los modelos mas nuevos debemos especificarle que modulos queremos apuntar
    # en el fine tunning.
    target_modules=['o_proj', 'qkv_proj', 'gate_up_proj', 'down_proj']
)

model = get_peft_model(model, config)
model

In [ ]:
print(model.get_memory_footprint()/1e6)

In [ ]:
train_p, tot_p = model.get_nb_trainable_parameters()
print(f'Parametros entrenables:      {train_p/1e6:.2f}M')
print(f'Parametros totales:          {tot_p/1e6:.2f}M')
print(f'% de parametros entrenables: {100*train_p/tot_p:.2f}%')

In [ ]:
#dataset = load_dataset("medalpaca/medical_meadow_wikidoc", split='train')
dataset = load_dataset("dvgodoy/yoda_sentences", split='train')
dataset

In [ ]:
#dataset = dataset.rename_column("input", "prompt")
#dataset = dataset.rename_column("output", "completion")
#dataset = dataset.remove_columns(["instruction"])
dataset = dataset.rename_column("sentence", "prompt")
dataset = dataset.rename_column("translation_extra", "completion")
dataset = dataset.remove_columns(["translation"])
dataset

In [ ]:
dataset[0]

In [ ]:
def format_dataset(examples):
  converted_sample = [
      {"role": "user", "content": examples["prompt"]},
      {"role": "assistant", "content": examples["completion"]},
  ]
  return {'messages': converted_sample}

In [ ]:
dataset = dataset.map(format_dataset).remove_columns(['prompt', 'completion'])
dataset[0]['messages']

In [ ]:
#split = dataset.train_test_split(test_size=0.9)
train = dataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(repo_id)
print(tokenizer.chat_template)

In [ ]:
print(tokenizer.apply_chat_template(conversation=train['messages'][32], tokenize=False))

In [ ]:
tokenizer.pad_token = tokenizer.unk_token
tokenizer.pad_token_id = tokenizer.unk_token_id

In [ ]:
sft = SFTConfig(
    gradient_checkpointing=True, # Con esto ahorramos un montón de memoria
    gradient_checkpointing_kwargs={"use_reentrant": False}, # Con las nuevas version de PyTorch necesitamos poner este argumento, mas información acá https://github.com/huggingface/transformers/issues/28536
    # Acumulacion del gradient y batch size
    gradient_accumulation_steps=1,
    per_device_train_batch_size=16,
    auto_find_batch_size=True, # Si el batch size que usas puede ocasionar un OOM (Out of Memory) lo dividimos en 2 hasta que funcione.

    #max_seq_length=768, # El tamaño de nuestra ventana de contexto.
    max_seq_length=64,
    packing=True,

    num_train_epochs=10,
    learning_rate=3e-4,

    optim='paged_adamw_8bit',

    logging_steps=10,
    logging_dir='./logs',
    output_dir='./phi3-mini-med-adapter',
    report_to='none'
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft,
    #train_dataset=split['train']
    train_dataset=train
)

In [ ]:
dl = trainer.get_train_dataloader()
batch = next(iter(dl))

In [ ]:
len(batch['input_ids'][0]), len(batch['labels'][0])

In [ ]:
trainer.train()

In [ ]:
def encode_prompt(tokenizer, sentence):
  sample = [{'role': 'user', 'content': sentence}]
  prompt = tokenizer.apply_chat_template(conversation=sample, tokenize=False, add_generation_prompt=True)
  return prompt

In [ ]:
#sentence = 'What are the historical background and symptoms of Candida-induced vulvovaginitis?'
sentence = 'The Force is strong in you!'
prompt = encode_prompt(tokenizer, sentence)
print(prompt)

In [ ]:
def inference(model, tokenizer, prompt, max_new_tokens=64, skip_special_tokens=False):
  tokenized_input = tokenizer(
      prompt, return_tensors='pt', add_special_tokens=False
  ).to(model.device)

  model.eval()

  gen_output = model.generate(**tokenized_input,
                              eos_token_id=tokenizer.eos_token_id,
                              max_new_tokens=max_new_tokens)

  output = tokenizer.batch_decode(gen_output, skip_special_tokens=skip_special_tokens)

  return output[0]

In [ ]:
print(inference(model, tokenizer, prompt))

In [ ]:
trainer.save_model('local-phi3-mini-yoda-adapter')

In [ ]:
!pip install huggingface_hub

In [ ]:
from huggingface_hub import login
login()

In [ ]:
trainer.push_to_hub()